### Probabilistic Graphical Models (PGM) & Naïve Bayes

**Tailored to `clinical_data.csv` :**
1. Inspects discrete variable mutual information with `y_multiclass` across clinical signs (`itch_true`, `grew_true`, `bleed_true`, `elevation_true`), anatomical `region_*`, `has_questionnaire`, and quantile-binned `age`, `mean_diameter`, and `symptom_count`.
2. Builds Laplace-smoothed **Conditional Probability Tables (CPTs)** $P(\text{diagnostic} \mid \text{Parents})$ and evaluates a factorized discrete graphical inference classifier on the held-out test set.
3. Evaluates **`GaussianNB`** with fold-safe imputation and `SMOTENC` imbalance handling under patient-grouped 5-fold CV.


In [11]:
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    mutual_info_score, accuracy_score, balanced_accuracy_score,
    f1_score, log_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


In [12]:
# 0. Configuration
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "src-code" else cwd

DATA_CANDIDATES = [
    "../Dataset/modelling/dataset_A_all.csv",
    "../Dataset/clinical_data.csv",
]
TARGET = "y_multiclass"
SEED = 42
N_FOLDS = 5
SAVE = True
OUT_DIR = PROJECT_ROOT / "Output" / "03_pgm_naive_bayes"

if SAVE:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["NEV", "ACK", "SEK", "BCC", "SCC", "MEL"]
CLASSES = list(range(len(LABELS)))
K = len(CLASSES)
MALIGNANT_IDX = [3, 4, 5]


## 1. Load Data & Patient-Grouped Stratified Split


In [13]:
def clean_and_engineer_from_raw(data: pd.DataFrame) -> pd.DataFrame:
    """Fallback preprocessor if notebook is run directly on clinical_data.csv."""
    d = data.copy()
    if "y_multiclass" not in d.columns:
        d["diagnostic"] = d["diagnostic"].astype(str).str.strip().str.upper()
        d = d[d["diagnostic"].isin(LABELS)].reset_index(drop=True)
        d["y_multiclass"] = d["diagnostic"].map({l: i for i, l in enumerate(LABELS)}).astype(int)
    if "is_cancer" not in d.columns:
        d["is_cancer"] = d["diagnostic"].isin({"BCC", "SCC", "MEL"}).astype(int)

    q_marker_cols = [c for c in ["fitspatrick", "diameter_1", "diameter_2", "gender"] if c in d.columns]
    if "has_questionnaire" not in d.columns:
        d["has_questionnaire"] = d[q_marker_cols].notna().any(axis=1).astype(int) if q_marker_cols else 1

    for col in ["age", "fitspatrick", "diameter_1", "diameter_2"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    if "diameter_missing" not in d.columns and {"diameter_1", "diameter_2"}.issubset(d.columns):
        d["diameter_missing"] = (d["diameter_1"].isna() | d["diameter_2"].isna()).astype(int)
    if "fitspatrick_missing" not in d.columns and "fitspatrick" in d.columns:
        d["fitspatrick_missing"] = d["fitspatrick"].isna().astype(int)

    if {"diameter_1", "diameter_2"}.issubset(d.columns):
        d1 = d["diameter_1"].clip(lower=0)
        d2 = d["diameter_2"].clip(lower=0)
        if "mean_diameter" not in d.columns:
            d["mean_diameter"] = 0.5 * (d1 + d2)
        if "diameter_ratio" not in d.columns:
            d_max = np.maximum(d1, d2)
            d_min = np.minimum(d1, d2).replace(0, np.nan)
            d["diameter_ratio"] = (d_max / d_min).fillna(1.0)
        if "log_area" not in d.columns:
            area_proxy = np.pi * (d1 / 2.0) * (d2 / 2.0)
            d["log_area"] = np.log1p(np.maximum(area_proxy, 0.0))

    symptom_cols = [c for c in ["itch", "grew", "hurt", "changed", "bleed", "elevation"] if c in d.columns]
    if symptom_cols:
        true_cols, unk_cols = [], []
        for sc in symptom_cols:
            s_clean = d[sc].fillna("UNK").astype(str).str.strip().str.upper()
            d[f"{sc}_true"] = (s_clean == "TRUE").astype(int)
            d[f"{sc}_unk"] = (s_clean == "UNK").astype(int)
            true_cols.append(f"{sc}_true")
            unk_cols.append(f"{sc}_unk")
        d = d.drop(columns=symptom_cols)
        d["symptom_count"] = d[true_cols].sum(axis=1).astype(float)
        d["symptom_unk_count"] = d[unk_cols].sum(axis=1).astype(float)
    else:
        if "symptom_count" not in d.columns:
            d["symptom_count"] = 0.0
        if "symptom_unk_count" not in d.columns:
            d["symptom_unk_count"] = 0.0

    if "age_x_symptoms" not in d.columns and {"age", "symptom_count"}.issubset(d.columns):
        d["age_x_symptoms"] = d["age"] * d["symptom_count"]
    if "symptom_per_decade" not in d.columns and {"age", "symptom_count"}.issubset(d.columns):
        d["symptom_per_decade"] = d["symptom_count"] / np.maximum(d["age"] / 10.0, 1.0)

    q_bool_cols = [
        c for c in [
            "smoke", "drink", "pesticide", "skin_cancer_history",
            "cancer_history", "has_piped_water", "has_sewage_system"
        ] if c in d.columns and d[c].dtype != int and d[c].dtype != np.int64
    ]
    for bc in q_bool_cols:
        s_str = d[bc].where(d[bc].notna(), "MISSING").astype(str).str.strip().str.upper()
        d[bc] = s_str.replace({"TRUE": "YES", "FALSE": "NO", "1": "YES", "0": "NO"})

    for bg_col in [c for c in ["background_father", "background_mother"] if c in d.columns]:
        s_bg = d[bg_col].fillna("MISSING").astype(str).str.strip().str.upper().replace({"BRASIL": "BRAZIL"})
        top_cats = {"POMERANIA", "GERMANY", "ITALY", "BRAZIL", "UNK", "MISSING"}
        d[bg_col] = np.where(s_bg.isin(top_cats), s_bg, "OTHER")

    for cat_c in [c for c in ["gender", "region"] if c in d.columns]:
        d[cat_c] = d[cat_c].fillna("MISSING").astype(str).str.strip().str.upper()

    cat_to_encode = [
        c for c in (q_bool_cols + ["gender", "region", "background_father", "background_mother"])
        if c in d.columns
    ]
    if cat_to_encode:
        d = pd.get_dummies(d, columns=cat_to_encode, dummy_na=False, dtype=int)

    return d


def load_project_dataset(candidates):
    path = next((Path(p) for p in candidates if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(f"None of the candidate data paths exist: {candidates}")
    raw = pd.read_csv(path)
    processed = clean_and_engineer_from_raw(raw)
    print(f"Loaded dataset from: {path} -> shape={processed.shape}")
    return processed, path

df, used_path = load_project_dataset(DATA_CANDIDATES)

NON_FEATURES = [
    "patient_id", "lesion_id", "img_id", "diagnostic",
    "y_multiclass", "is_cancer", "has_questionnaire", "biopsed"
]
ALL_FEATS = [c for c in df.columns if c not in NON_FEATURES]
NUMERIC = [
    c for c in [
        "age", "fitspatrick", "diameter_1", "diameter_2",
        "mean_diameter", "diameter_ratio", "log_area",
        "symptom_count", "symptom_unk_count", "age_x_symptoms", "symptom_per_decade"
    ] if c in ALL_FEATS
]
FEATS = NUMERIC + [c for c in ALL_FEATS if c not in NUMERIC]
CAT_IDX = list(range(len(NUMERIC), len(FEATS)))

splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(splitter.split(df, df[TARGET], df["patient_id"]))
train = df.iloc[tr_idx].reset_index(drop=True)
test = df.iloc[te_idx].reset_index(drop=True)

X_train, y_train, g_train = train[FEATS].copy(), train[TARGET].values, train["patient_id"].values
X_test, y_test, g_test = test[FEATS].copy(), test[TARGET].values, test["patient_id"].values
assert len(set(g_train) & set(g_test)) == 0


Loaded dataset from: ..\Dataset\modelling\dataset_A_all.csv -> shape=(2298, 85)


## 2. Probabilistic Graphical Model (PGM): Dependency Ranking, CPTs & Posterior Inference


In [14]:
def build_discrete_pgm_tables(X_tr: pd.DataFrame, X_te: pd.DataFrame, num_cols: list):
    d_tr = pd.DataFrame(index=X_tr.index)
    d_te = pd.DataFrame(index=X_te.index)
    for col in [c for c in ["age", "mean_diameter", "symptom_count", "log_area"] if c in num_cols]:
        med_val = X_tr[col].median()
        s_tr = X_tr[col].fillna(med_val)
        s_te = X_te[col].fillna(med_val)
        _, bins = pd.qcut(s_tr, q=3, retbins=True, duplicates="drop")
        bins[0], bins[-1] = -np.inf, np.inf
        d_tr[f"{col}_q3"] = pd.cut(s_tr, bins=bins, labels=False).astype(int)
        d_te[f"{col}_q3"] = pd.cut(s_te, bins=bins, labels=False).astype(int)
    for col in [c for c in X_tr.columns if c not in num_cols]:
        d_tr[col] = X_tr[col].fillna(0).astype(int)
        d_te[col] = X_te[col].fillna(0).astype(int)
    return d_tr, d_te

disc_tr, disc_te = build_discrete_pgm_tables(X_train, X_test, NUMERIC)

mi_map = {col: mutual_info_score(disc_tr[col], y_train) for col in disc_tr.columns}
mi_series = pd.Series(mi_map).sort_values(ascending=False)
print("=== Top 15 Discrete PGM Nodes by Mutual Information with y_multiclass ===")
print(mi_series.head(15).round(4))

parent_cols = [c for c in ["age_q3", "symptom_count_q3"] if c in disc_tr.columns]
if len(parent_cols) == 2:
    alpha = 1.0
    cpt_records = []
    combos = disc_tr[parent_cols].drop_duplicates().sort_values(parent_cols)
    for _, r in combos.iterrows():
        mask = (disc_tr[parent_cols[0]] == r[parent_cols[0]]) & (disc_tr[parent_cols[1]] == r[parent_cols[1]])
        yc = y_train[mask]
        cnts = np.bincount(yc, minlength=K)
        probs = (cnts + alpha) / (cnts.sum() + alpha * K)
        rec = {parent_cols[0]: int(r[parent_cols[0]]), parent_cols[1]: int(r[parent_cols[1]]), "n_train": int(cnts.sum())}
        for idx_c, lbl in enumerate(LABELS):
            rec[f"P({lbl})"] = round(float(probs[idx_c]), 3)
        cpt_records.append(rec)
    cpt_table = pd.DataFrame(cpt_records)
    print("\n=== Empirical CPT: P(diagnostic | age_q3, symptom_count_q3) ===")
    print(cpt_table.to_string(index=False))
    if SAVE:
        cpt_table.to_csv(OUT_DIR / "pgm_cpt_age_symptoms.csv", index=False)

top_nodes = list(mi_series.head(6).index)
alpha = 1.0
prior = (np.bincount(y_train, minlength=K) + alpha) / (len(y_train) + alpha * K)
log_post = np.tile(np.log(prior), (len(disc_te), 1))

for node in top_nodes:
    states = sorted(disc_tr[node].unique())
    S = len(states)
    state_to_i = {s: i for i, s in enumerate(states)}
    lik = np.zeros((K, S))
    for c in range(K):
        yc_mask = (y_train == c)
        nc = yc_mask.sum()
        for s_i, s_val in enumerate(states):
            cnt = ((disc_tr[node] == s_val) & yc_mask).sum()
            lik[c, s_i] = (cnt + alpha) / (nc + alpha * S)
    for row_i, val in enumerate(disc_te[node]):
        if val in state_to_i:
            log_post[row_i] += np.log(lik[:, state_to_i[val]] + 1e-15)

log_post -= log_post.max(axis=1, keepdims=True)
pgm_proba_test = np.exp(log_post)
pgm_proba_test /= pgm_proba_test.sum(axis=1, keepdims=True)
pgm_pred_test = pgm_proba_test.argmax(1)

present_te = np.unique(y_test)
pgm_f1 = f1_score(y_test, pgm_pred_test, labels=present_te, average="macro", zero_division=0)
print(
    f"\nFactorized PGM (top {len(top_nodes)} nodes) Test -> "
    f"Acc: {accuracy_score(y_test, pgm_pred_test):.3f} | "
    f"BalAcc: {balanced_accuracy_score(y_test, pgm_pred_test):.3f} | "
    f"Macro-F1: {pgm_f1:.3f}"
)


=== Top 15 Discrete PGM Nodes by Mutual Information with y_multiclass ===
background_mother_MISSING      0.3343
background_father_MISSING      0.3310
drink_MISSING                  0.3223
smoke_MISSING                  0.3223
diameter_missing               0.3223
fitspatrick_missing            0.3223
pesticide_MISSING              0.3223
gender_MISSING                 0.3223
skin_cancer_history_MISSING    0.3223
has_sewage_system_MISSING      0.3223
cancer_history_MISSING         0.3223
has_piped_water_MISSING        0.3223
symptom_count_q3               0.1658
smoke_NO                       0.1630
itch_true                      0.1393
dtype: float64

=== Empirical CPT: P(diagnostic | age_q3, symptom_count_q3) ===
 age_q3  symptom_count_q3  n_train  P(NEV)  P(ACK)  P(SEK)  P(BCC)  P(SCC)  P(MEL)
      0                 0      280   0.283   0.465   0.066   0.140   0.010   0.035
      0                 1      257   0.327   0.224   0.080   0.281   0.053   0.034
      0                 2  

## 3. Naïve Bayes (`GaussianNB`) with Fold-Safe Imputation & `SMOTENC`


In [15]:
def make_nb_pipe(strategy="none", var_smoothing=1e-9):
    if strategy == "smote":
        def med_strat(y):
            cnts = Counter(y)
            med = int(np.median(list(cnts.values())))
            return {k: med for k, v in cnts.items() if v < med}
        return ImbPipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("smote", SMOTENC(categorical_features=CAT_IDX, sampling_strategy=med_strat, k_neighbors=3, random_state=SEED)),
            ("scale", StandardScaler()),
            ("gnb", GaussianNB(var_smoothing=var_smoothing)),
        ])
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("gnb", GaussianNB(var_smoothing=var_smoothing)),
    ])

cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
nb_summary = []

for strat in ["none", "smote"]:
    fold_f1, fold_bacc = [], []
    for tr_f, va_f in cv.split(X_train, y_train, g_train):
        m = make_nb_pipe(strategy=strat, var_smoothing=1e-9)
        m.fit(X_train.iloc[tr_f], y_train[tr_f])
        p_va = m.predict(X_train.iloc[va_f])
        pres = np.unique(y_train[va_f])
        fold_f1.append(f1_score(y_train[va_f], p_va, labels=pres, average="macro", zero_division=0))
        fold_bacc.append(balanced_accuracy_score(y_train[va_f], p_va))

    full_m = make_nb_pipe(strategy=strat, var_smoothing=1e-9)
    full_m.fit(X_train, y_train)
    proba_te = np.zeros((len(X_test), K))
    proba_te[:, full_m.classes_] = full_m.predict_proba(X_test)
    pred_te = proba_te.argmax(1)

    nb_summary.append({
        "strategy": strat,
        "cv_macro_f1_mean": float(np.mean(fold_f1)),
        "cv_macro_f1_std": float(np.std(fold_f1)),
        "cv_bal_acc_mean": float(np.mean(fold_bacc)),
        "test_macro_f1": float(f1_score(y_test, pred_te, labels=present_te, average="macro", zero_division=0)),
        "test_bal_acc": float(balanced_accuracy_score(y_test, pred_te)),
        "test_log_loss": float(log_loss(y_test, np.clip(proba_te, 1e-15, 1), labels=CLASSES)),
    })

nb_df = pd.DataFrame(nb_summary)
print("=== GaussianNB Grouped CV & Test Summary ===")
print(nb_df.round(4).to_string(index=False))

if SAVE:
    nb_df.to_csv(OUT_DIR / "naive_bayes_summary.csv", index=False)


=== GaussianNB Grouped CV & Test Summary ===
strategy  cv_macro_f1_mean  cv_macro_f1_std  cv_bal_acc_mean  test_macro_f1  test_bal_acc  test_log_loss
    none            0.2672           0.0255           0.4142         0.2498        0.4323        20.6425
   smote            0.2754           0.0189           0.4121         0.2369        0.3518        20.4823
